#Instalations

In [ ]:
!pip -q install -U --no-cache-dir \
  "transformers==4.46.3" \
  "pandas==2.2.2" \
  "datasets" "tqdm"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 150.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 273.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 272.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 168.3 MB/s eta 0:00:00


#Imports and connection to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import torch
import pickle
import pandas as pd
import datasets
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
)


Mounted at /content/drive


#Config

In [ ]:
BASE_DIR = "/content/drive/MyDrive/PyTester_Colab"

# DATASET_PATH = f"{BASE_DIR}/data/processed_apps_test.csv"
DATASET_PATH = f"{BASE_DIR}/data/apps_test.csv"
OUTPUT_BASE = f"{BASE_DIR}/inference_outputs_apps_test"

MODELS = {
    "original_pytester": f"{BASE_DIR}/models/original",
}
for _, model in MODELS.items():
  if os.path.exists(model):
    print(f"{model} ok")
  else:
    print(f"{model} nie ok")

INPUT_COLUMN = "prompt_testcase"
BATCH_SIZE = 8
BEAM_SIZE = 5
MAX_NEW_TOKENS = 500
MAX_LENGTH = 512
PRED_MODE = "multi"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_BASE, exist_ok=True)

/content/drive/MyDrive/PyTester_Colab/models/original ok


#Load Data

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Liczba przykładów testowych: {len(df)}")

test_inputs_text = list(df[INPUT_COLUMN])

dataset = datasets.Dataset.from_dict({
    "text": test_inputs_text
})


Liczba przykładów testowych: 1000


#Function and run

In [ ]:
def detect_weight_format(model_path: str) -> str:
    """
    Zwraca: 'safetensors', 'bin' albo 'hub'
    """
    if os.path.isdir(model_path):
        if os.path.exists(os.path.join(model_path, "model.safetensors")):
            return "safetensors"
        if os.path.exists(os.path.join(model_path, "pytorch_model.bin")):
            return "bin"
    return "hub"


def select_torch_dtype(device: str):
    """
    Dobiera optymalny dtype pod Colaba
    """
    if device != "cuda":
        return torch.float32

    # bf16 jeśli GPU wspiera
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16

    return torch.float16

def run_inference(model_name: str, model_path: str):
    print(f"\n=== INFERENCE: {model_name} ===")

    weight_format = detect_weight_format(model_path)
    torch_dtype = select_torch_dtype(DEVICE)

    print(f"→ Wykryty format wag: {weight_format}")
    print(f"→ Używany dtype: {torch_dtype}")

    # Tokenizer – zawsze bazowy CodeT5
    tokenizer = AutoTokenizer.from_pretrained(
        "Salesforce/codet5-large",
        truncation_side="left",
        padding_side="right"
    )

    # Model
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        local_files_only=True
    )

    model.to(DEVICE)
    model.eval()

    texts = list(dataset["text"])

    predictions = []

    for text in tqdm(texts):
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=BEAM_SIZE,
                num_return_sequences=BEAM_SIZE,
                pad_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        predictions.append(decoded)

    out_dir = f"{OUTPUT_BASE}/{model_name}"
    os.makedirs(out_dir, exist_ok=True)

    with open(f"{out_dir}/predictions.pkl", "wb") as f:
        pickle.dump(predictions, f)

    with open(f"{out_dir}/predictions.txt", "w") as f:
        for preds in predictions:
            f.write(preds[0] + "\n")

    print(f"Zapisano wyniki do: {out_dir}")

    del model
    del tokenizer
    torch.cuda.empty_cache()




#Inference

In [ ]:
for name, path in MODELS.items():
    run_inference(name, path)


=== INFERENCE: original_pytester ===
→ Wykryty format wag: bin
→ Używany dtype: torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Some weights of the model checkpoint at /content/drive/MyDrive/PyTester_Colab/models/original were not used when initializing T5ForConditionalGeneration: ['v_head.summary.bias', 'v_head.summary.weight']
- This IS expected if you are initializing T5ForConditionalGeneration from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing T5ForConditionalGeneration from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|██████████| 1000/1000 [1:11:11<00:00,  4.27s/it]

Zapisano wyniki do: /content/drive/MyDrive/PyTester_Colab/inference_outputs_apps_test/original_pytester


In [ ]:
import google.colab
google.colab.runtime.unassign()

In [ ]:
# =========================
# SINGLE PROMPT INFERENCE
# =========================

def load_model_for_chat(model_path: str):
    """
    Ładuje tokenizer i model tylko raz (do interaktywnego użycia)
    """
    weight_format = detect_weight_format(model_path)
    torch_dtype = select_torch_dtype(DEVICE)

    print(f"→ Format wag: {weight_format}")
    print(f"→ dtype: {torch_dtype}")

    tokenizer = AutoTokenizer.from_pretrained(
        "Salesforce/codet5-large",
        truncation_side="left",
        padding_side="right"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        local_files_only=True
    )

    model.to(DEVICE)
    model.eval()

    return tokenizer, model


def generate_single_response(
    prompt: str,
    tokenizer,
    model,
    max_new_tokens: int = 300,
    beam_size: int = 5,
    show_all_beams: bool = False
):
    """
    Generuje odpowiedź modelu na pojedynczy prompt
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=beam_size,
            num_return_sequences=beam_size,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    print("\n================ MODEL RESPONSE ================\n")

    if show_all_beams:
        for i, resp in enumerate(decoded):
            print(f"[Beam {i+1}]\n{resp}\n{'-'*50}")
    else:
        print(decoded[0])

    return decoded


In [ ]:
MODELS = {
    "original_pytester": f"{BASE_DIR}/models/original",
}

MODEL_NAME = "sft"
MODEL_PATH = MODELS[MODEL_NAME]

tokenizer, model = load_model_for_chat(MODEL_PATH)

prompt = '''def call_solution():\n    r"""\n    There is a popular apps named “Exbook” like “Facebook”. To sign up in this app , You have to make a strong password with more than 3 digits and less than 10 digits . But I am a pro hacker and so I make a Exbook hacking site .
You need to login in this site to hack exbook account and then you will get a portal. You can give any user exbook login link using this site and when anyone login into exbook using your link ,you can see his/her password .\n
But I made a mistake and so you cannot find original password in your portal . The portal showing you by adding two in every digit .
So , now you have to find out the original password of an user if I give you the password which is showing in your portal .\n    -----Input:-----\n
The first line contains a single integer t (1 ≤ t ≤ 1000) — the number of test cases.\n    The first line of each test case contains a single integer n which is showing in your portal .
Mind it , every digit of n is greater than one .\n    -----Output:-----\n    Print , original password of user .\n    -----Sample Input:-----\n    2\n    3527\n    47269\n
-----Sample Output:-----\n    1305\n    25047\n    """\n    pass\n
# Generate assertion statements for the function that cover different execution paths including edge cases'''

prompt2 = '''def reverse_words(sentence: str) -> str:
	"""
	Reverses the order of words in a given sentence.
	Multiple spaces are reduced to a single one, and leading/trailing spaces are removed.
	If input is not a string, an empty string is returned.
	Parameters:
	sentence : str - The input sentence to reverse.
	Returns - str - Sentence with reversed word order.
	Example:
	reverse_words(Ala has a cat) = cat a has Ala
	"""
	pass
#Generate assertion statements for the function that cover different execution paths including edge cases'''

prompt3 = '''def calc_median(values: list[float]) -> float | None:
	"""
	Calculates the median value from a list of floats.
	For even count, returns the average of two middle numbers.
	Parameters:
	values : list - List of floats.
	Returns - float - Median value rounded to 2 decimals, or None if invalid input.
	Example:
	calc_median([1.0, 2.0, 3.0, 4.0]) = 2.5
	"""
	pass
#Generate assertion statements for the function that cover different execution paths including edge cases'''

generate_single_response(
    prompt,
    tokenizer,
    model,
    show_all_beams=False
)

→ Format wag: safetensors
→ dtype: torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]


================ MODEL RESPONSE ================

assert call_solution('2\n3527\n47269') == '134\n25047'


["assert call_solution('2\\n3527\\n47269') == '134\\n25047'",
 "assert call_solution('2\\n3527\\n47269') == '130\\n25047'",
 "assert call_solution('2\\n3527\\n47269') == '133\\n25047'",
 "assert call_solution('2\\n3527\\n47269') == '1305\\n25047'",
 "assert call_solution('2\\n3527\\n47269') == '113\\n25047'"]